# Capture TCP Stream to MP4

This notebook connects to the ESP32 TCP camera stream, decodes each JPEG frame, and saves the session as an `.mp4` file.

Stream format used here matches the rest of this repo:
- 4-byte big-endian frame length
- JPEG payload

Defaults are aligned with the current Jetson runtime config: `192.168.4.1:8080`.

If you connect successfully but no frames arrive, the ESP32 usually accepted the socket before the camera queue started producing frames. This notebook now waits gracefully and prints idle status instead of raising a raw traceback.

In [ ]:
import os
import socket
import struct
import time
from datetime import datetime

import cv2
import numpy as np

# Connection settings
HOST = os.getenv("GLASSES_HOST", "000.000.0.0")
PORT = int(os.getenv("GLASSES_PORT", "0000"))

# Output settings
OUTPUT_DIR = "captures/mp4"
OUTPUT_NAME_PREFIX = "tcp_capture"
OUTPUT_FPS = 24.0  # Playback FPS. Tune if you want a faster/slower saved video.
FOURCC = "mp4v"

# Capture limits. Leave as None to capture until you stop the cell.
MAX_FRAMES = None
MAX_SECONDS = 1800

# Optional local preview window. Set to True if this notebook is running where OpenCV GUI windows are supported.
SHOW_PREVIEW = True
PREVIEW_WINDOW = "TCP Stream Preview"

CONNECT_TIMEOUT_SEC = 10.0
READ_TIMEOUT_SEC = 1.0
READ_IDLE_LIMIT_SEC = None  # Set to a number of seconds if you want the notebook to stop waiting.
IDLE_STATUS_INTERVAL_SEC = 5.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Ready to capture from {HOST}:{PORT}")
print(f"Saved videos will go to: {os.path.abspath(OUTPUT_DIR)}")

Ready to capture from 192.168.4.1:8080
Saved videos will go to: /Users/erickortega/ai_glasses/ML Notebook/captures/mp4


In [5]:
def recv_exact(sock, nbytes, label="stream data"):
    """Receive exactly nbytes from the TCP stream or return None on EOF.

    Short socket read timeouts are treated as idle periods so the notebook can
    wait for frames without crashing as soon as the ESP32 pauses.
    """
    buf = bytearray()
    wait_started = time.time()
    last_status_at = wait_started

    while len(buf) < nbytes:
        try:
            chunk = sock.recv(nbytes - len(buf))
        except socket.timeout:
            now = time.time()
            waited = now - wait_started
            if now - last_status_at >= IDLE_STATUS_INTERVAL_SEC:
                print(f"Still waiting for {label} from {HOST}:{PORT} ({waited:.1f}s idle)")
                last_status_at = now
            if READ_IDLE_LIMIT_SEC is not None and waited >= READ_IDLE_LIMIT_SEC:
                raise TimeoutError(
                    f"No {label} received for {waited:.1f}s after connecting to {HOST}:{PORT}. "
                    "The ESP32 accepted the TCP socket but has not produced frames yet."
                )
            continue

        if not chunk:
            return None

        buf.extend(chunk)

    return bytes(buf)


def build_output_path(output_dir, prefix):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return os.path.join(output_dir, f"{prefix}_{stamp}.mp4")


def open_stream(host, port):
    sock = socket.create_connection((host, port), timeout=CONNECT_TIMEOUT_SEC)
    sock.settimeout(READ_TIMEOUT_SEC)
    return sock

In [6]:
sock = None
writer = None
output_path = None
frame_count = 0
decode_failures = 0
started_at = time.time()
first_frame_at = None

try:
    sock = open_stream(HOST, PORT)
    print(f"Connected to {HOST}:{PORT}")

    while True:
        header = recv_exact(sock, 4, label="frame header")
        if header is None:
            print("Stream ended before the next frame header.")
            break

        frame_len = struct.unpack(">I", header)[0]
        payload = recv_exact(sock, frame_len, label=f"frame payload ({frame_len} bytes)")
        if payload is None:
            print("Stream ended in the middle of a frame payload.")
            break

        frame = cv2.imdecode(np.frombuffer(payload, np.uint8), cv2.IMREAD_COLOR)
        if frame is None:
            decode_failures += 1
            print(f"Skipping frame {frame_count + 1}: JPEG decode failed")
            continue

        if writer is None:
            height, width = frame.shape[:2]
            output_path = build_output_path(OUTPUT_DIR, OUTPUT_NAME_PREFIX)
            fourcc = cv2.VideoWriter_fourcc(*FOURCC)
            writer = cv2.VideoWriter(output_path, fourcc, OUTPUT_FPS, (width, height))
            if not writer.isOpened():
                raise RuntimeError(
                    f"Could not open VideoWriter for {output_path}. Try a different FOURCC."
                )
            print(f"Writing {width}x{height} video to {output_path}")

        writer.write(frame)
        if first_frame_at is None:
            first_frame_at = time.time()
        frame_count += 1

        total_elapsed = time.time() - started_at
        capture_elapsed = time.time() - first_frame_at if first_frame_at is not None else 0.0

        if SHOW_PREVIEW:
            cv2.imshow(PREVIEW_WINDOW, frame)
            if (cv2.waitKey(1) & 0xFF) == ord("q"):
                print("Preview window requested stop.")
                break

        if frame_count % 25 == 0:
            live_fps = frame_count / capture_elapsed if capture_elapsed > 0 else 0.0
            print(f"Captured {frame_count} frames in {capture_elapsed:.1f}s ({live_fps:.2f} fps input)")

        if MAX_FRAMES is not None and frame_count >= MAX_FRAMES:
            print(f"Reached MAX_FRAMES={MAX_FRAMES}")
            break

        if MAX_SECONDS is not None and total_elapsed >= MAX_SECONDS:
            print(f"Reached MAX_SECONDS={MAX_SECONDS}")
            break

except TimeoutError as exc:
    print(f"Timed out waiting for stream data: {exc}")
    print("Check the ESP32 serial logs for camera init or capture failures, or leave READ_IDLE_LIMIT_SEC=None to wait longer.")
except OSError as exc:
    print(f"Socket error while capturing: {exc}")
except KeyboardInterrupt:
    print("Capture interrupted from the notebook.")
finally:
    if sock is not None:
        sock.close()
    if writer is not None:
        writer.release()
    if SHOW_PREVIEW:
        cv2.destroyAllWindows()

elapsed_total = time.time() - started_at
capture_elapsed = time.time() - first_frame_at if first_frame_at is not None else 0.0
avg_input_fps = frame_count / capture_elapsed if capture_elapsed > 0 else 0.0

print(f"Frames written: {frame_count}")
print(f"Decode failures: {decode_failures}")
print(f"Elapsed: {elapsed_total:.2f}s")
print(f"Average input FPS: {avg_input_fps:.2f}")
if output_path:
    print(f"Saved MP4: {os.path.abspath(output_path)}")
else:
    print("No video file was created because no decodable frames were received.")

Connected to 192.168.4.1:8080
Writing 320x240 video to captures/mp4/tcp_capture_20260403_000703.mp4
Captured 25 frames in 4.5s (5.61 fps input)
Captured 50 frames in 9.6s (5.23 fps input)
Captured 75 frames in 14.5s (5.18 fps input)
Captured 100 frames in 19.5s (5.14 fps input)
Captured 125 frames in 24.5s (5.11 fps input)
Captured 150 frames in 29.6s (5.07 fps input)
Captured 175 frames in 34.5s (5.08 fps input)
Captured 200 frames in 39.5s (5.07 fps input)
Captured 225 frames in 44.6s (5.05 fps input)
Captured 250 frames in 49.6s (5.04 fps input)
Captured 275 frames in 54.5s (5.05 fps input)
Captured 300 frames in 59.6s (5.04 fps input)
Captured 325 frames in 64.6s (5.03 fps input)
Captured 350 frames in 69.5s (5.04 fps input)
Captured 375 frames in 74.5s (5.04 fps input)
Captured 400 frames in 79.5s (5.03 fps input)
Captured 425 frames in 85.0s (5.00 fps input)
Captured 450 frames in 90.0s (5.00 fps input)
Captured 475 frames in 95.2s (4.99 fps input)
Captured 500 frames in 100.0s (